# Lab 04b: A Poda Inteligente (Pruning & Batching)

**Disciplina:** Sistemas Distribuídos para IA (1CXIA - S10)
**Objetivo:** Aplicar técnicas de Poda (Pruning) para reduzir o tamanho de modelos e Loteamento (Batching) para maximizar o Throughput.
**Ambiente:** Google Colab (Exclusivo).

## 🚀 Instruções de Acesso ao Google Colab

Este laboratório utiliza GPU para aceleração de inferência. Siga os passos:

1. Acesse: [https://colab.research.google.com/](https://colab.research.google.com/) e crie um **Novo Notebook**.
2. **ATIVAR GPU:** Vá em `Ambiente de Execução` -> `Alterar o tipo de ambiente de execução`.
3. Selecione **T4 GPU** e clique em **Salvar**.

## 1. Setup e Baseline do Modelo
Vamos carregar uma ResNet-18 pré-treinada e verificar seu tamanho original em disco.

In [ ]:
import torch
import torch.nn.utils.prune as prune
import torchvision.models as models
import time
import os

# Carregando ResNet-18 original
model = models.resnet18(weights="IMAGENET1K_V1").to("cuda")

# Verificando tamanho original em disco (Simulado via salvamento)
torch.save(model.state_dict(), "resnet18_original.pth")
size_orig = os.path.getsize("resnet18_original.pth") / (1024**2)
print(f"💾 Tamanho Original: {size_orig:.2f} MB")

## 2. Passo 1: A Poda (Unstructured Pruning)
Vamos remover **50% dos pesos** das camadas convolucionais que possuem os menores valores absolutos (L1 Unstructured).

In [ ]:
# Aplicando poda de 50% em todas as camadas convolucionais
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        prune.l1_unstructured(module, name='weight', amount=0.5)
        prune.remove(module, 'weight') # Torna a poda permanente

# Verificando a esparsidade (Quantos pesos agora são ZERO)
total_zeros = 0
total_params = 0
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        total_zeros += torch.sum(module.weight == 0)
        total_params += module.weight.nelement()

print(f"✂️ Esparsidade: {100. * float(total_zeros) / total_params:.2f}% dos neurônios removidos.")

## 3. Passo 2: Batching (Processamento Individual vs. Lote)
Aqui está o segredo do Throughput em sistemas distribuídos. Vamos processar 64 imagens de duas formas.

In [ ]:
# Simulando 64 imagens (3 canais, 224x224 pixels)
data = torch.randn(64, 3, 224, 224).to("cuda")

# TESTE A: Processamento Individual (Batch Size = 1)
print("⏱️ Iniciando Teste A (Individual)...")
start_a = time.time()
for i in range(64):
    _ = model(data[i].unsqueeze(0))
end_a = time.time()
latencia_individual = (end_a - start_a) / 64

# TESTE B: Processamento em Lote (Batch Size = 64)
print("⏱️ Iniciando Teste B (Lote)...")
start_b = time.time()
_ = model(data)
end_b = time.time()
latencia_lote = (end_b - start_b) / 64

print(f"\n🚀 Resultados:")
print(f"⏱️ Tempo Médio por Imagem (Individual): {latencia_individual:.4f}s")
print(f"⏱️ Tempo Médio por Imagem (Lote): {latencia_lote:.4f}s")
print(f"🔥 Ganho de Vazão (Throughput): {latencia_individual / latencia_lote:.2f}x")

## 4. Desafio Tier 2: Poda Radical
Tente alterar o código acima para realizar uma poda de **90%**. 
1. Verifique se o modelo ainda gera saídas válidas (não gera erro).
2. O ganho de velocidade mudou com a poda maior?

## 5. 🔑 Gabarito do Desafio (Resultado Esperado)

Se você realizou a **Poda Radical (90%)**, deve ter observado os seguintes fenômenos:

1. **Acurácia:** O modelo ainda gera saídas válidas, mas o reconhecimento de imagem falha drasticamente. Com 90% dos pesos zerados, o modelo perde as "características finas" das imagens.
2. **Velocidade:** No PyTorch padrão, **a velocidade NÃO muda com a poda**. O PyTorch continua multiplicando matrizes densas (cheias de zeros). Para ganhar velocidade real, seriam necessárias bibliotecas de **Cálculo Esparso (Sparse Kernels)**.
3. **Batching:** A vazão aumenta no lote porque a GPU é um **processador massivamente paralelo**. No lote, a GPU preenche todos os seus núcleos e processa tudo de uma vez.